# WP2: SLAKE budget run (one budget + seed per run)
GPU on, Internet on, secrets GITHUB_TOKEN + HF_TOKEN attached. Edit SETTINGS in Cell B: BUDGET, SEED, SAMPLING ('image' or 'question'), TEST_MODE. Each run writes only its own folder, so two copies can run at the same time. First do a TEST_MODE = True commit.

In [ ]:
# CELL A - INSTALL (first cell; nothing heavy imported before it). Stops immediately if an install fails.
import sys, subprocess
print("Python", sys.version.split()[0])
def pip(args, must=True):
    r = subprocess.run(f"{sys.executable} -m pip {args}", shell=True, capture_output=True, text=True)
    if r.returncode != 0 and must:
        raise RuntimeError("pip failed: " + args + "\n" + (r.stdout + r.stderr)[-2500:])
    return r
# No numpy/pandas here on purpose: upgrading numpy inside a running kernel breaks it. qwen-vl-utils without the decord extra (video only).
pip("install -q -U transformers accelerate peft 'qwen-vl-utils==0.0.8' datasets huggingface_hub matplotlib tabulate")
pip("install -q --force-reinstall 'pillow<11'")
pip("uninstall -y torchao", must=False)      # Kaggle's old torchao breaks newer peft; not used here
print("Install done.")

In [ ]:
# CELL B - SETTINGS for this run, secrets, Hugging Face login, GitHub clone
import os, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login

# ---------------- SETTINGS (change these for each run) ----------------
BUDGET     = 0.10     # fraction of TRAINING IMAGES: 0.05, 0.10 or 0.25   (50% / 100% need the resume version, coming later)
SEED       = 42       # 42, 123 or 2024 (changes which images are chosen AND the training order)
EPOCHS     = 3
LR         = 2e-4
GRAD_ACCUM = 8
SAMPLING   = "image"   # "image" = budget counted in training IMAGES (all questions of chosen images); "question" = budget counted in QUESTIONS (like the pilot)
TEST_MODE  = False    # True = 16 training questions, 1 epoch, 20 eval questions (about 10-15 min total); results go to a *_test folder
# ----------------------------------------------------------------------
assert BUDGET in (0.05, 0.10, 0.25), "this version supports 5%, 10%, 25% (up to about 4.3 h of training)"
assert SAMPLING in ("image", "question")
RUN_ID = f"slake_{'img' if SAMPLING == 'image' else 'q'}{round(BUDGET * 100)}_s{SEED}" + ("_test" if TEST_MODE else "")
if TEST_MODE: EPOCHS = 1
GITHUB_REPO, HF_REPO_NAME = "sanshubhGupta/med-vqa-lora-btp", "med-vqa-qwen2.5-vl-slake"

GH_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(GH_TOKEN, "***")[-2000:])
REPO = Path("/kaggle/working/med-vqa-lora-btp")
sh('git config --global user.email "sanshubh08gupta@gmail.com"'); sh('git config --global user.name "sanshubhGupta"')
if not REPO.exists():
    sh(f"cd /kaggle/working && git clone https://{GH_TOKEN}@github.com/{GITHUB_REPO}.git")
    sh(f"cd {REPO} && git remote set-url origin https://github.com/{GITHUB_REPO}.git")

login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
hf_api = HfApi(); HF_REPO_ID = f"{hf_api.whoami()['name']}/{HF_REPO_NAME}"
assert "results_full/predictions_baseline_zero_shot.csv" in hf_api.list_repo_files(HF_REPO_ID), "zero-shot predictions not found on the Hub"
if SAMPLING == "image":
    assert (REPO / "configs" / f"image_budgets_seed{SEED}.json").exists(), "budget file missing from the GitHub repo (run WP0 first)"
print("RUN_ID:", RUN_ID, "| HF repo:", HF_REPO_ID)

def push_folder(folder, path_in_repo=None, msg="Upload"):
    try:
        hf_api.upload_folder(folder_path=str(folder), repo_id=HF_REPO_ID, repo_type="model", path_in_repo=path_in_repo, commit_message=msg)
        print("Pushed to", f"https://huggingface.co/{HF_REPO_ID}", f"({path_in_repo})" if path_in_repo else "")
    except Exception as e:
        print("HF push FAILED (continuing):", repr(e)[:300])

In [ ]:
# CELL C - data and the image-level training budget for this run
import gc, json, random, re, time, zipfile
import numpy as np, pandas as pd, torch
from PIL import Image
from datasets import load_dataset
from huggingface_hub import snapshot_download, hf_hub_download

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
PROMPT   = "Answer the question in a few words only. Question: {q}"
MAX_PIX, MIN_PIX = 256 * 28 * 28, 64 * 28 * 28
OUT = Path(f"/kaggle/working/{RUN_ID}"); OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

slake = load_dataset("BoKelvin/SLAKE")
slake_dir = Path(snapshot_download(repo_id="BoKelvin/SLAKE", repo_type="dataset"))
extract_dir = slake_dir / "unzipped_imgs"
if not extract_dir.exists():
    with zipfile.ZipFile(slake_dir / "imgs.zip") as z:
        z.extractall(extract_dir)
def get_img(row):
    return Image.open(Path(extract_dir) / "imgs" / row["img_name"]).convert("RGB")
train_en = slake["train"].filter(lambda r: r["q_lang"] == "en")
test_en  = slake["test"].filter(lambda r: r["q_lang"] == "en")

if SAMPLING == "image":
    cfg = json.load(open(REPO / "configs" / f"image_budgets_seed{SEED}.json"))
    entry = next(v for k, v in cfg.items() if abs(float(k) - BUDGET) < 1e-9)
    train_idx = sorted(entry["question_idx"]); n_images = len(entry["images"])
    assert set(train_en[i]["img_name"] for i in train_idx) <= set(entry["images"]), "question/image mismatch in budget file"
else:   # question-level budget, same code as the pilot: stratified by answer type, random questions
    _tmp = pd.DataFrame({"idx": range(len(train_en)), "answer_type": list(train_en["answer_type"])})
    train_idx = sorted(_tmp.groupby("answer_type", group_keys=False).sample(frac=BUDGET, random_state=SEED)["idx"].tolist())
    n_images = len({train_en[i]["img_name"] for i in train_idx})
print(f"{SAMPLING}-level budget {BUDGET:.0%} seed {SEED}: {n_images} images, {len(train_idx)} questions")
if TEST_MODE:
    train_idx = train_idx[:16]; print("TEST_MODE: using", len(train_idx), "training questions")
n_eval = len(test_en) if not TEST_MODE else 20
print("English test:", len(test_en), "| evaluating", n_eval)

In [ ]:
# CELL D - model, input builder (training and inference), metric
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIX, max_pixels=MAX_PIX)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

def build_inputs(image, question, answer=None):
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": image, "min_pixels": MIN_PIX, "max_pixels": MAX_PIX},
        {"type": "text", "text": PROMPT.format(q=question)}]}]
    prompt_text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(msgs)
    if answer is None:
        return processor(text=[prompt_text], images=imgs, videos=vids, return_tensors="pt")
    enc = processor(text=[prompt_text + str(answer) + "<|im_end|>"], images=imgs, videos=vids, return_tensors="pt")
    prompt_len = processor(text=[prompt_text], images=imgs, videos=vids, return_tensors="pt")["input_ids"].shape[1]
    labels = enc["input_ids"].clone(); labels[:, :prompt_len] = -100          # loss only on the answer
    enc["labels"] = labels
    return enc

def norm(s):
    s = re.sub(r"[^\w\s]", " ", str(s).lower().strip()); s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(t[:-1] if len(t) > 3 and t.endswith("s") else t for t in s.split())
def score(pred, gt):
    p, g = norm(pred), norm(gt); gt_tok, pred_tok = set(g.split()), set(p.split())
    return p == g, (len(gt_tok & pred_tok) / len(gt_tok) if gt_tok else 0.0)

_r = train_en[train_idx[0]]; _c = build_inputs(get_img(_r), _r["question"], _r["answer"])
_kept = _c["labels"][0][_c["labels"][0] != -100]
print("Unmasked label text:", repr(processor.tokenizer.decode(_kept)), "| ground truth:", _r["answer"])

# ---- environment record + reproducibility check against the saved zero-shot answers ----
import sys, importlib.metadata as md
def _v(p):
    try: return md.version(p)
    except Exception: return "?"
ENV = "python=" + sys.version.split()[0] + ";" + ";".join(f"{p}={_v(p)}" for p in ("torch", "transformers", "peft", "accelerate", "qwen-vl-utils", "datasets", "huggingface_hub", "pillow", "numpy"))
print("ENV:", ENV)
@torch.no_grad()
def _zs_predict(image, question):
    inputs = build_inputs(image, question).to(model.device)
    out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
_ref = pd.read_csv(hf_hub_download(HF_REPO_ID, "results_full/predictions_baseline_zero_shot.csv", repo_type="model"), keep_default_na=False, dtype=str)
_n = 10 if TEST_MODE else 100
model.eval()
_same = sum(norm(_zs_predict(get_img(test_en[i]), test_en[i]["question"])) == norm(_ref["pred"][i]) for i in range(_n))
ZS_AGREE = round(_same / _n, 3)
print(f"Reproducibility: this environment reproduces {_same}/{_n} of the saved zero-shot answers ({ZS_AGREE:.0%}). Expect about 95%+; a much lower value means the setup differs from the pilot.")

In [ ]:
# CELL E - LoRA training (same recipe as the first run). Adapter + loss pushed to the Hub after every epoch.
from peft import LoraConfig, get_peft_model
from transformers import get_cosine_schedule_with_warmup

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
model.print_trainable_parameters(); model.config.use_cache = False

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
total_steps = max(1, EPOCHS * len(train_idx) // GRAD_ACCUM)
sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * total_steps)), total_steps)
print(f"{len(train_idx)} questions x {EPOCHS} epochs -> about {total_steps} optimizer steps")

losses, running, step, t0 = [], 0.0, 0, time.time()
torch.cuda.reset_peak_memory_stats(); model.train()
for ep in range(EPOCHS):
    order = train_idx[:]; random.Random(SEED * 1000 + ep).shuffle(order)
    opt.zero_grad(set_to_none=True); running = 0.0
    for k, j in enumerate(order):
        r = train_en[j]
        enc = build_inputs(get_img(r), r["question"], r["answer"]).to(model.device)
        loss = model(**enc).loss / GRAD_ACCUM
        if not torch.isfinite(loss): raise RuntimeError(f"Loss is {loss.item()} at epoch {ep+1}, example {k}")
        loss.backward(); running += loss.item()
        if (k + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1; losses.append(running); running = 0.0
            if step % 10 == 0 or step == 1:
                el = (time.time() - t0) / 60
                print(f"epoch {ep+1} step {step}/{total_steps} loss {losses[-1]:.4f} | {el:.1f} min, about {el / step * (total_steps - step):.0f} min left")
    ckpt = OUT / f"ckpt_ep{ep + 1}"; model.save_pretrained(ckpt)
    json.dump(losses, open(ckpt / "train_loss.json", "w"))
    push_folder(ckpt, path_in_repo=f"runs/{RUN_ID}/checkpoints/ep{ep + 1}", msg=f"{RUN_ID}: checkpoint after epoch {ep + 1}")

train_minutes = round((time.time() - t0) / 60, 1); peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 1)
json.dump(losses, open(OUT / "train_loss.json", "w"))
final = OUT / "adapter"; model.save_pretrained(final)
push_folder(final, path_in_repo=f"runs/{RUN_ID}/adapter", msg=f"{RUN_ID}: final adapter")
print("training minutes:", train_minutes, "| peak GPU GB:", peak_gb)

In [ ]:
# CELL F - evaluate on all English test questions, image-clustered CIs, comparison with zero-shot, registry row
model.config.use_cache = True; model.eval()

@torch.no_grad()
def predict(image, question):
    inputs = build_inputs(image, question).to(model.device)
    out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

rows, t1 = [], time.time()
for i in range(n_eval):
    r = test_en[i]; pred = predict(get_img(r), r["question"]); exact, rec = score(pred, r["answer"])
    rows.append(dict(i=i, img_name=r["img_name"], question=r["question"], gt=str(r["answer"]), pred=pred,
                     answer_type=r["answer_type"], content_type=r["content_type"], exact=float(exact), recall=rec))
    if (i + 1) % 100 == 0: print(f"[eval] {i + 1}/{n_eval} ({(time.time() - t1) / 60:.1f} min)")
d = pd.DataFrame(rows); d.to_csv(OUT / "predictions.csv", index=False)

def _per_image(df, value, mask=None):
    imgs = df["img_name"].unique(); x = df if mask is None else df[mask]
    g = x.groupby("img_name")[value].agg(["sum", "count"]).reindex(imgs, fill_value=0)
    return g["sum"].values.astype(float), g["count"].values.astype(float)
def cluster_boot_mean(df, value, mask=None, n_boot=5000, seed=0):
    S, N = _per_image(df, value, mask); rng = np.random.RandomState(seed)
    idx = rng.randint(0, len(S), (n_boot, len(S))); num, den = S[idx].sum(1), N[idx].sum(1); est = num[den > 0] / den[den > 0]
    return (S.sum() / N.sum() if N.sum() else float("nan")), np.percentile(est, 2.5), np.percentile(est, 97.5), int(N.sum())

base = pd.read_csv(hf_hub_download(HF_REPO_ID, "results_full/predictions_baseline_zero_shot.csv", repo_type="model"),
                   keep_default_na=False, dtype={"gt": str, "pred": str, "question": str}).iloc[:n_eval].reset_index(drop=True)
assert (base["gt"].values == d["gt"].values).all(), "row order differs from the zero-shot predictions"
d["zero_shot_exact"] = base["exact"].astype(float).values; d["diff"] = d["exact"] - d["zero_shot_exact"]

subsets = [("overall", None), ("CLOSED", d.answer_type == "CLOSED"), ("OPEN", d.answer_type == "OPEN")] + \
          [(c, d.content_type == c) for c in sorted(d.content_type.unique())]
table = []
for name, mask in subsets:
    row = {"subset": name}
    for col, lab in (("zero_shot_exact", "zero_shot"), ("exact", "this_run"), ("diff", "gain")):
        pt, lo, hi, n = cluster_boot_mean(d, col, mask); row[lab] = f"{pt:.3f} [{lo:.3f}, {hi:.3f}]"; row["n"] = n
    table.append(row)
tab = pd.DataFrame(table); tab.to_csv(OUT / "by_category_with_ci.csv", index=False)
print(f"RESULTS {RUN_ID} ({n_images} images, {len(train_idx)} questions, n_eval={n_eval})"); print(tab.to_string(index=False))

closed, opn = d[d.answer_type == "CLOSED"], d[d.answer_type == "OPEN"]
ov = cluster_boot_mean(d, "exact")
summary = dict(run_id=RUN_ID, sampling=SAMPLING, dataset="SLAKE-en", backbone=MODEL_ID, budget=BUDGET, seed=SEED, epochs=EPOCHS, lr=LR, rank=8,
               n_images=n_images, n_questions=len(train_idx), train_minutes=train_minutes, peak_gpu_gb=peak_gb, n_eval=n_eval,
               closed_exact=round(closed.exact.mean(), 4), open_exact=round(opn.exact.mean(), 4), open_recall=round(opn.recall.mean(), 4),
               overall_exact=round(d.exact.mean(), 4), overall_ci_lo=round(ov[1], 4), overall_ci_hi=round(ov[2], 4),
               zs_repro_agree=ZS_AGREE, env=ENV)
json.dump(summary, open(OUT / "summary.json", "w"), indent=2); print(summary)
import subprocess as _sp
open(OUT / "pip_freeze.txt", "w").write(_sp.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)

In [ ]:
# CELL G - record the result: Hub backup, per-run note, GitHub (skipped in TEST_MODE). Each run only writes its OWN folder, so runs can safely overlap.
import datetime
push_folder(OUT, path_in_repo=f"runs/{RUN_ID}/results", msg=f"{RUN_ID}: results")
if TEST_MODE:
    print("TEST_MODE: skipping GitHub. Set TEST_MODE = False for the real run.")
else:
    dest = REPO / "results" / "runs" / RUN_ID; dest.mkdir(parents=True, exist_ok=True)
    for f in ("predictions.csv", "by_category_with_ci.csv", "summary.json", "train_loss.json", "pip_freeze.txt"):
        sh(f"cp {OUT}/{f} {dest}/")
    with open(dest / "NOTE.md", "w") as fh:
        fh.write(f"## {datetime.date.today()} - {RUN_ID}\n- {n_images} training images / {len(train_idx)} questions, {EPOCHS} epochs, lr {LR}, seed {SEED}; "
                 f"train {train_minutes} min, peak {peak_gb} GB\n- test (n={n_eval}): overall {summary['overall_exact']} "
                 f"[{summary['overall_ci_lo']}, {summary['overall_ci_hi']}], closed {summary['closed_exact']}, open {summary['open_exact']}\n- env: {ENV}\n")
    url = f"https://{GH_TOKEN}@github.com/{GITHUB_REPO}.git"
    sh(f"cd {REPO} && git add results && git commit -m 'Run {RUN_ID}'; git pull --rebase {url} main; git push {url} HEAD:main")
print("DONE", RUN_ID)